# Automated Insight Generation — Data Analytics Notebook
This notebook demonstrates the step-by-step logic used in our application:
1. Data loading and validation
2. Month-over-month trend detection
3. IQR statistical outlier detection
4. Pearson correlation matrix
5. Automated plain-English insight generation

In [ ]:
import pandas as pd
import numpy as np

# Step 1: Load dataset
df = pd.read_csv("data/healthcare_data.csv")
print("Dataset Shape:", df.shape)
print("
Missing values:")
print(df.isna().sum())
df.head(6)

In [ ]:
# Step 2: Trend Detection (Month-over-month percentage change)
threshold = 10.0  # 10% cutoff
indicators = ["anc_coverage", "institutional_delivery", "immunization", "high_risk_cases"]
trends = []

for district, group in df.groupby("district"):
    group = group.sort_values("month").reset_index(drop=True)
    if len(group) >= 2:
        for ind in indicators:
            for i in range(1, len(group)):
                prev = group.loc[i - 1, ind]
                curr = group.loc[i, ind]
                if prev > 0:
                    pct = ((curr - prev) / prev) * 100.0
                    if abs(pct) >= threshold:
                        trends.append({
                            "district": district,
                            "indicator": ind,
                            "period": group.loc[i, "month"],
                            "prev": prev,
                            "curr": curr,
                            "pct_change": round(pct, 1)
                        })

pd.DataFrame(trends)

In [ ]:
# Step 3: Outlier Detection (IQR Method: Q1 - 1.5*IQR to Q3 + 1.5*IQR)
iqr_mult = 1.5
outliers = []

for ind in indicators:
    vals = df[ind]
    q1 = vals.quantile(0.25)
    q3 = vals.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - (iqr_mult * iqr)
    upper = q3 + (iqr_mult * iqr)
    mean_val = vals.mean()
    
    for _, row in df.iterrows():
        val = row[ind]
        if val < lower or val > upper:
            outliers.append({
                "district": row["district"],
                "indicator": ind,
                "period": row["month"],
                "value": val,
                "mean": round(mean_val, 1),
                "lower_bound": round(lower, 1),
                "upper_bound": round(upper, 1)
            })

pd.DataFrame(outliers)

In [ ]:
# Step 4: Pearson Correlation Matrix
corr_matrix = df[indicators].corr(method="pearson").round(2)
corr_matrix

In [ ]:
# Step 5: Automated Insight Generation
# Building plain-English sentences directly from computed numbers
insights = []
for t in trends:
    dir_word = "dropped" if t["pct_change"] < 0 else "increased"
    insights.append({
        "insight_id": f"INS-{len(insights)+1:04d}",
        "type": "trend",
        "district": t["district"],
        "explanation": f"{t['indicator']} in {t['district']} {dir_word} by {abs(t['pct_change'])}% (from {t['prev']} to {t['curr']})"
    })

pd.DataFrame(insights)